# 06 - Regenerate every table and figure (P8.7)

Runs on a clean Colab runtime: installs dependencies, builds the manifest from the released checkpoints, verifies every SHA-256 and model load, then regenerates the protocol tables. B3 is installed here because PyBullet is only needed by this notebook.

In [ ]:
!pip -q install stable-baselines3==2.9.0 scipy pandas matplotlib pyyaml
!pip -q install gym-pybullet-drones

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
print('repo root:', ROOT)


In [ ]:
from training.manifest import build_manifest, verify_manifest
from pathlib import Path
manifest = build_manifest(ROOT / 'runs', ROOT / 'results' / 'manifest.json')
problems = verify_manifest(manifest, ROOT / 'runs', load=True)
print(len(manifest['runs']), 'checkpoints; problems:', problems)

In [ ]:
!python scripts/collect_guidance_distributions.py --workers 4
!python scripts/plot_guidance.py


In [ ]:
!python scripts/evaluate_guidance.py --episodes 100 --workers 4

In [ ]:
for platform in ('quadcopter', 'fixed_wing', 'missile'):
    !python scripts/evaluate_guidance_arms.py --episodes 100 --workers 4 --a3 runs/p7_{platform}_a3_s0/checkpoint.zip --a4 runs/p7_{platform}_a4_s0/checkpoint.zip


In [ ]:
!python scripts/evaluate_fc.py --rl runs/p7_fc_asym_s0/checkpoint.zip

In [ ]:
!python scripts/evaluate_closed_loop.py --episodes 100 --workers 4 --guidance-policy runs/p7_quadcopter_a3_s0/checkpoint.zip

In [ ]:
from pathlib import Path
for path in sorted((ROOT / 'results' / 'tables').glob('*.csv')):
    print('\n==', path.name)
    print(path.read_text(encoding='utf-8')[:1200])

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
for path in sorted((ROOT / 'results' / 'tables').glob('guidance_*_evasive.csv')):
    df = pd.read_csv(path)
    fig, ax = plt.subplots(figsize=(10, 4))
    for arm, g in df.groupby('arm'):
        ax.bar(g['behavior'] + ' ' + arm, g['median_miss'], alpha=.6, label=arm)
    ax.set(ylabel='median miss distance (m)', title=path.stem)
    ax.tick_params(axis='x', rotation=90, labelsize=7)
    fig.tight_layout()
    fig.savefig(ROOT / 'results' / 'figures' / f'{path.stem}.png', dpi=150)
    plt.close(fig)